# Sports RAG - Retriever Phase Experiments

This notebook implements the retrieval phase exactly for your plan:
- BM25 on fixed chunks
- BM25 on recursive chunks
- BM25 on semantic chunks
- Hybrid (BM25 fixed + dense fixed namespace) with RRF
- Hybrid (BM25 recursive + dense recursive namespace) with RRF
- Hybrid (BM25 semantic + dense semantic namespace) with RRF

It now builds a 100-question evaluation bank from the three chunk JSON files, including diverse question types and unanswerable prompts for abstention testing.
It evaluates retrievers with quality + latency metrics, compares winner QA latency, and tracks abstention accuracy in dev mode.

## 1) Install Dependencies

In [1]:
!pip -q install rank-bm25 sentence-transformers pinecone pandas numpy tqdm transformers accelerate openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 14.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 19.6 MB/s eta 0:00:00


## 2) Imports and Global Config

In [ ]:
import os
import json
import re
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from pinecone import Pinecone
from transformers import pipeline

# Runtime mode
DEV_MODE = True

# Evaluation sizing
TARGET_EVAL_QUESTIONS = 100
UNANSWERABLE_QUESTIONS = 15
QA_EVAL_SIZE = 40 if DEV_MODE else TARGET_EVAL_QUESTIONS

# Retrieval hyperparameters
TOP_K = 10
RRF_K = 60
DENSE_TOP_K = 50

# Re-ranking hyperparameters
RERANK_CANDIDATES_TOP_K = 30
QA_CONTEXT_TOP_K = 5

# LLM-as-judge configuration
JUDGE_PROVIDER = "groq"  # "groq" or "openai"
JUDGE_MODEL_NAME = "llama-3.1-8b-instant"  # Higher TPD limit (500k vs 100k)
JUDGE_EVAL_SIZE = 10 if DEV_MODE else 30

# Ablation sizing
ABLATION_EVAL_SIZE = 8 if DEV_MODE else 15

# Embedding model for dense retrieval queries (good quality + Kaggle-friendly size)
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

# Cross-encoder model for final re-ranking (stronger quality on Kaggle GPU)
CROSS_ENCODER_MODEL_NAME = "BAAI/bge-reranker-base"

# QA model for final winner-vs-winner answer comparison
QA_MODEL_NAME = "google/flan-t5-base"

# Kaggle Secrets support for API keys.
try:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()

    pinecone_secret = user_secrets.get_secret("PINECONE_API_KEY")
    if pinecone_secret:
        os.environ["PINECONE_API_KEY"] = pinecone_secret
        print("Loaded PINECONE_API_KEY from Kaggle Secrets.")

    openai_secret = user_secrets.get_secret("OPENAI_API_KEY")
    if openai_secret:
        os.environ["OPENAI_API_KEY"] = openai_secret
        print("Loaded OPENAI_API_KEY from Kaggle Secrets.")

    groq_secret = user_secrets.get_secret("GROQ_API_KEY")
    if groq_secret:
        os.environ["GROQ_API_KEY"] = groq_secret
        print("Loaded GROQ_API_KEY from Kaggle Secrets.")
except Exception:
    # Local environment or missing Kaggle Secrets; env vars can still be set manually.
    pass

print(f"Imports and config loaded. DEV_MODE={DEV_MODE}")

Loaded PINECONE_API_KEY from Kaggle Secrets.
Imports and config loaded.


## 3) Locate Input Files and Load Chunks

In [3]:
def resolve_chunk_dir() -> Path:
    cwd = Path.cwd()

    # Preferred local/Kaggle working directory names
    candidate_dirs = [
        cwd / "chunks_BM25",
        cwd / "Keyword Chunks",
        cwd.parent / "chunks_BM25",
        cwd.parent / "Keyword Chunks",
    ]

    for d in candidate_dirs:
        if d.exists():
            return d

    # Kaggle dataset mount fallback: /kaggle/input/<dataset>/...
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        # First, look for a directory named chunks_BM25 anywhere under /kaggle/input
        for d in kaggle_input.rglob("chunks_BM25"):
            if d.is_dir():
                return d

        # Then, look for any directory that contains all three required files
        required = {"chunks_fixed.json", "chunks_recursive.json", "chunks_semantic.json"}
        for fixed_file in kaggle_input.rglob("chunks_fixed.json"):
            parent = fixed_file.parent
            names = {p.name for p in parent.glob("*.json")}
            if required.issubset(names):
                return parent

    raise FileNotFoundError(
        "Could not find chunk JSON folder. Expected one of: "
        "./chunks_BM25, ./Keyword Chunks, ../chunks_BM25, ../Keyword Chunks, "
        "or a Kaggle input directory containing chunks_fixed.json/chunks_recursive.json/chunks_semantic.json."
    )

CHUNK_DIR = resolve_chunk_dir()
PROJECT_ROOT = CHUNK_DIR.parent

chunk_paths = {
    "fixed": CHUNK_DIR / "chunks_fixed.json",
    "recursive": CHUNK_DIR / "chunks_recursive.json",
    "semantic": CHUNK_DIR / "chunks_semantic.json",
}

chunks_by_strategy = {}
for strategy, path in chunk_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        chunks_by_strategy[strategy] = json.load(f)

print(f"Chunk directory: {CHUNK_DIR}")
for strategy, chunks in chunks_by_strategy.items():
    print(f"{strategy}: {len(chunks)} chunks")

# Build id->chunk lookup per strategy
chunk_lookup = {
    strategy: {item["id"]: item for item in items}
    for strategy, items in chunks_by_strategy.items()
}

Chunk directory: /kaggle/input/datasets/muhammadhamzaarif/keyword-chunks
fixed: 4872 chunks
recursive: 4860 chunks
semantic: 4998 chunks


## 4) Build BM25 Indexes (Fixed, Recursive, Semantic)

In [4]:
def bm25_tokenize(text: str):
    return re.findall(r"[A-Za-z0-9]+", text.lower())

bm25_indexes = {}
for strategy, chunks in chunks_by_strategy.items():
    tokenized_corpus = [bm25_tokenize(c["text"]) for c in chunks]
    bm25_indexes[strategy] = BM25Okapi(tokenized_corpus)

print("Built BM25 indexes for fixed/recursive/semantic.")

Built BM25 indexes for fixed/recursive/semantic.


## 5) BM25 and Dense Retrieval Helpers

In [5]:
def bm25_retrieve(query: str, strategy: str, top_k: int = TOP_K):
    chunks = chunks_by_strategy[strategy]
    bm25 = bm25_indexes[strategy]

    q_tokens = bm25_tokenize(query)
    scores = bm25.get_scores(q_tokens)
    ranked_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in ranked_idx:
        chunk = chunks[idx]
        results.append({
            "id": chunk["id"],
            "score": float(scores[idx]),
            "text": chunk.get("text", ""),
            "source": chunk.get("source", ""),
            "url": chunk.get("url", ""),
            "strategy": strategy,
        })
    return results

def init_dense_components(index_name: str = "sports-rag"):
    pinecone_api_key = os.environ.get("PINECONE_API_KEY")
    if not pinecone_api_key:
        raise EnvironmentError("Set PINECONE_API_KEY before running dense retrieval.")

    pc = Pinecone(api_key=pinecone_api_key)
    index = pc.Index(index_name)
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    return index, embed_model

def dense_retrieve(query: str, namespace: str, strategy: str, index, embed_model, top_k: int = TOP_K):
    q_vec = embed_model.encode(query, normalize_embeddings=True).tolist()

    response = index.query(
        vector=q_vec,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True,
    )

    local_lookup = chunk_lookup[strategy]
    results = []

    for match in response.get("matches", []):
        match_id = match.get("id")
        metadata = match.get("metadata", {}) or {}
        fallback = local_lookup.get(match_id, {})

        results.append({
            "id": match_id,
            "score": float(match.get("score", 0.0)),
            "text": metadata.get("text", fallback.get("text", "")),
            "source": metadata.get("source", fallback.get("source", "")),
            "url": metadata.get("url", fallback.get("url", "")),
            "strategy": strategy,
        })
    return results

## 6) Reciprocal Rank Fusion (RRF) and Hybrid Retriever

In [6]:
def reciprocal_rank_fusion(result_lists, rrf_k: int = RRF_K, top_k: int = TOP_K):
    fused_scores = defaultdict(float)
    payload = {}

    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            doc_id = item["id"]
            fused_scores[doc_id] += 1.0 / (rrf_k + rank)
            if doc_id not in payload:
                payload[doc_id] = item

    ranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    fused = []
    for doc_id, score in ranked:
        item = payload[doc_id].copy()
        item["score"] = float(score)
        fused.append(item)
    return fused

def hybrid_retrieve(query: str, strategy: str, namespace: str, index, embed_model, top_k: int = TOP_K):
    sparse_hits = bm25_retrieve(query, strategy=strategy, top_k=DENSE_TOP_K)
    dense_hits = dense_retrieve(
        query,
        namespace=namespace,
        strategy=strategy,
        index=index,
        embed_model=embed_model,
        top_k=DENSE_TOP_K,
    )
    return reciprocal_rank_fusion([sparse_hits, dense_hits], rrf_k=RRF_K, top_k=top_k)

## 7) Cross-Encoder Re-Ranking Helpers

In [7]:
def init_cross_encoder(model_name: str = CROSS_ENCODER_MODEL_NAME):
    return CrossEncoder(model_name)

def cross_encoder_rerank(query: str, candidates, cross_encoder, top_k: int = TOP_K):
    if not candidates:
        return []

    # Cross-encoder scores each (query, document) pair directly.
    pairs = [[query, c.get("text", "")] for c in candidates]
    scores = cross_encoder.predict(pairs)

    reranked = []
    for item, ce_score in zip(candidates, scores):
        obj = item.copy()
        obj["rerank_score"] = float(ce_score)
        reranked.append(obj)

    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]

## 8) Sanity Check Queries

In [8]:
sample_query = "Who won the 2022 FIFA World Cup final?"

print("BM25 fixed sample:")
for r in bm25_retrieve(sample_query, strategy="fixed", top_k=3):
    print(r["id"], "|", r["score"])

BM25 fixed sample:
fixed_Argentina_national_football_team_1 | 15.051653894026892
fixed_Kylian_Mbappe_3 | 14.83107891177794
fixed_Manchester_City_FC_20 | 14.566597227609407


## 9) Build 100-Question Evaluation Set (Diverse + Abstention)

This section auto-builds 100 questions from the fixed/recursive/semantic chunk files.
It mixes multiple question styles and adds unanswerable questions for abstention evaluation.

Each item includes:
- question: string
- relevant_chunk_ids: list of acceptable IDs (empty for unanswerable)
- question_type: category label
- expect_abstain: whether the model should say "I don't know"

In [ ]:
def parse_entity_from_chunk_id(chunk_id: str, strategy_prefix: str):
    pattern = rf"^{strategy_prefix}_(.*)_\d+$"
    m = re.match(pattern, chunk_id)
    if not m:
        return None
    return m.group(1)

def pretty_entity(entity_slug: str):
    return entity_slug.replace("_", " ").strip()

def infer_question_type(entity_name: str, entity_text: str):
    name = entity_name.lower()
    text = entity_text.lower()

    if "national football team" in name or "national football team" in text:
        return "country-team"
    if any(k in name for k in ["cup", "league", "bundesliga", "eredivisie", "libertadores", "champions"]):
        return "competition"
    if any(k in text for k in ["footballer", "former player", "striker", "midfielder", "goalkeeper"]) or len(entity_name.split()) <= 3:
        # Lightweight heuristic: many person pages are short names and mention footballer/player in opening text.
        if any(k in text for k in ["footballer", "player", "manager"]):
            return "person"
    if any(k in name for k in ["f.c", "fc", "afc", "club", "milan", "arsenal", "chelsea", "dortmund", "barcelona", "bayern", "ajax"]):
        return "club"
    return "general"

def generate_question_variants(entity_name: str, qtype: str):
    if qtype == "country-team":
        return [
            f"Which country does {entity_name} represent?",
            f"{entity_name} represents which nation?",
            f"In international football, what country is {entity_name} associated with?",
        ]
    if qtype == "competition":
        return [
            f"What competition is {entity_name}?",
            f"How is {entity_name} categorized in football?",
            f"What type of tournament or league is {entity_name}?",
        ]
    if qtype == "club":
        return [
            f"What club is {entity_name}?",
            f"In football, what is {entity_name} known as?",
            f"How would you describe {entity_name} in club football?",
        ]
    if qtype == "person":
        return [
            f"Who is {entity_name}?",
            f"Give a short football profile of {entity_name}.",
            f"What is {entity_name} known for in football?",
        ]
    return [
        f"What is {entity_name}?",
        f"Give a brief overview of {entity_name} in football.",
        f"In football context, what does {entity_name} refer to?",
    ]

# Build entity alignment across fixed/recursive/semantic
id_by_strategy_by_entity = defaultdict(dict)
text_by_entity = {}

for strategy in ["fixed", "recursive", "semantic"]:
    for item in chunks_by_strategy[strategy]:
        entity_slug = parse_entity_from_chunk_id(item["id"], strategy)
        if not entity_slug:
            continue
        if entity_slug not in id_by_strategy_by_entity or strategy not in id_by_strategy_by_entity[entity_slug]:
            id_by_strategy_by_entity[entity_slug][strategy] = item["id"]
            text_by_entity.setdefault(entity_slug, item.get("text", ""))

common_entities = [
    e for e, m in id_by_strategy_by_entity.items()
    if {"fixed", "recursive", "semantic"}.issubset(set(m.keys()))
]
common_entities = sorted(common_entities)

answerable_target = TARGET_EVAL_QUESTIONS - UNANSWERABLE_QUESTIONS
answerable_questions = []
used_questions = set()

for entity_slug in common_entities:
    entity_name = pretty_entity(entity_slug)
    qtype = infer_question_type(entity_name, text_by_entity.get(entity_slug, ""))
    candidates = generate_question_variants(entity_name, qtype)
    relevant_ids = [
        id_by_strategy_by_entity[entity_slug]["fixed"],
        id_by_strategy_by_entity[entity_slug]["recursive"],
        id_by_strategy_by_entity[entity_slug]["semantic"],
    ]

    for q in candidates:
        q_norm = q.strip().lower()
        if q_norm in used_questions:
            continue
        used_questions.add(q_norm)
        answerable_questions.append({
            "question": q,
            "relevant_chunk_ids": relevant_ids,
            "question_type": qtype,
            "expect_abstain": False,
        })
        if len(answerable_questions) >= answerable_target:
            break
    if len(answerable_questions) >= answerable_target:
        break

if len(answerable_questions) < answerable_target:
    raise ValueError(
        f"Could only generate {len(answerable_questions)} answerable questions; need {answerable_target}."
    )

unanswerable_questions = [
    "Who won the UEFA Euro 2024 Golden Boot?",
    "What was Lionel Messi's salary at Inter Miami in 2025?",
    "Who scored the fastest goal in FIFA World Cup 2030?",
    "Which team won the 2026 AFC Champions League final?",
    "What is the transfer fee for Kylian Mbappe in 2026?",
    "Who is the top scorer of the Saudi Pro League 2025 season?",
    "Which club signed Jude Bellingham in summer 2027?",
    "What was the exact xG of the 2024 Champions League final?",
    "Who won the Ballon d'Or in 2026?",
    "What was the attendance of the 2025 Club World Cup final?",
    "Which city hosted the 2028 FIFA World Cup opening match?",
    "Who was the referee for the 2029 Copa Libertadores final?",
    "What was Pep Guardiola's win rate in 2025?",
    "Which footballer announced retirement in January 2027?",
    "What was the VAR decision timeline in the 2026 UCL final?",
]

if len(unanswerable_questions) != UNANSWERABLE_QUESTIONS:
    raise ValueError("UNANSWERABLE_QUESTIONS does not match the hardcoded list length.")

unanswerable_items = [
    {
        "question": q,
        "relevant_chunk_ids": [],
        "question_type": "unanswerable",
        "expect_abstain": True,
    }
    for q in unanswerable_questions
]

evaluation_questions = answerable_questions + unanswerable_items
random.seed(42)
random.shuffle(evaluation_questions)

if len(evaluation_questions) != TARGET_EVAL_QUESTIONS:
    raise ValueError(
        f"Expected {TARGET_EVAL_QUESTIONS} questions, found {len(evaluation_questions)}."
    )

evaluation_questions_df = pd.DataFrame(evaluation_questions)
print("Total questions:", len(evaluation_questions))
print("Answerable:", int((~evaluation_questions_df["expect_abstain"]).sum()))
print("Unanswerable:", int(evaluation_questions_df["expect_abstain"].sum()))
print("Question type mix:")
display(evaluation_questions_df["question_type"].value_counts().rename_axis("question_type").to_frame("count"))
evaluation_questions_df.head(10)

Current #questions: 25
All 25 football-only evaluation questions loaded with real chunk IDs.


## 10) Evaluation Metrics: Recall@5, Recall@10, MRR@10

In [ ]:
import time

def compute_recall_at_k(pred_ids, relevant_ids, k):
    top_pred = pred_ids[:k]
    hits = len(set(top_pred).intersection(set(relevant_ids)))
    return 1.0 if hits > 0 else 0.0

def compute_mrr_at_k(pred_ids, relevant_ids, k=10):
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(pred_ids[:k], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def evaluate_retriever(eval_data, retriever_fn, top_k=10):
    rows = []
    for item in tqdm(eval_data, desc="Evaluating"):
        q = item["question"]
        rel = item.get("relevant_chunk_ids", [])
        expect_abstain = bool(item.get("expect_abstain", False) or len(rel) == 0)

        retrieval_start = time.perf_counter()
        results = retriever_fn(q, top_k)
        retrieval_time_sec = time.perf_counter() - retrieval_start

        pred_ids = [r["id"] for r in results]

        rows.append({
            "question": q,
            "question_type": item.get("question_type", "unknown"),
            "expect_abstain": expect_abstain,
            "recall@5": compute_recall_at_k(pred_ids, rel, 5) if not expect_abstain else np.nan,
            "recall@10": compute_recall_at_k(pred_ids, rel, 10) if not expect_abstain else np.nan,
            "mrr@10": compute_mrr_at_k(pred_ids, rel, 10) if not expect_abstain else np.nan,
            "retrieval_time_sec": retrieval_time_sec,
        })

    df = pd.DataFrame(rows)
    answerable_df = df[~df["expect_abstain"]].copy()
    unanswerable_df = df[df["expect_abstain"]].copy()

    return {
        "Recall@5": answerable_df["recall@5"].mean() if not answerable_df.empty else np.nan,
        "Recall@10": answerable_df["recall@10"].mean() if not answerable_df.empty else np.nan,
        "MRR@10": answerable_df["mrr@10"].mean() if not answerable_df.empty else np.nan,
        "Answerable Count": len(answerable_df),
        "Unanswerable Count": len(unanswerable_df),
        "Avg Retrieval Time (s)": df["retrieval_time_sec"].mean(),
        "P95 Retrieval Time (s)": df["retrieval_time_sec"].quantile(0.95),
        "details": df,
    }

## 11) Run All 6 Retrieval Experiments

In [ ]:
experiments = {
    "bm25_fixed": lambda q, k: bm25_retrieve(q, "fixed", top_k=k),
    "bm25_recursive": lambda q, k: bm25_retrieve(q, "recursive", top_k=k),
    "bm25_semantic": lambda q, k: bm25_retrieve(q, "semantic", top_k=k),
}

dense_ready = bool(os.environ.get("PINECONE_API_KEY"))
if dense_ready:
    index, embed_model = init_dense_components(index_name="sports-rag")
    experiments.update({
        "hybrid_fixed": lambda q, k: hybrid_retrieve(q, "fixed", "fixed", index, embed_model, top_k=k),
        "hybrid_recursive": lambda q, k: hybrid_retrieve(q, "recursive", "recursive", index, embed_model, top_k=k),
        "hybrid_semantic": lambda q, k: hybrid_retrieve(q, "semantic", "semantic", index, embed_model, top_k=k),
    })
else:
    print("PINECONE_API_KEY not set: running BM25-only experiments.")

all_results = []
retrieval_detail_by_experiment = {}
for name, retriever_fn in experiments.items():
    metrics = evaluate_retriever(evaluation_questions, retriever_fn, top_k=10)
    all_results.append({
        "experiment": name,
        "answerable_questions": metrics["Answerable Count"],
        "unanswerable_questions": metrics["Unanswerable Count"],
        "Recall@5": metrics["Recall@5"],
        "Recall@10": metrics["Recall@10"],
        "MRR@10": metrics["MRR@10"],
        "avg_retrieval_time_sec": metrics["Avg Retrieval Time (s)"],
        "p95_retrieval_time_sec": metrics["P95 Retrieval Time (s)"],
    })
    retrieval_detail_by_experiment[name] = metrics["details"]

results_df = pd.DataFrame(all_results).sort_values(["MRR@10", "avg_retrieval_time_sec"], ascending=[False, True])
results_df

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Evaluating: 100%|██████████| 25/25 [00:01<00:00, 14.89it/s]


,experiment,Recall@5,Recall@10,MRR@10
5,hybrid_semantic,0.80,0.92,0.716111
3,hybrid_fixed,0.80,0.88,0.713000
4,hybrid_recursive,0.76,0.88,0.711667
2,bm25_semantic,0.60,0.68,0.311556
0,bm25_fixed,0.48,0.60,0.281714
1,bm25_recursive,0.48,0.64,0.281048


## 12) Winner Selection (1 Best Sparse + 1 Best Hybrid)

In [12]:
sparse_df = results_df[results_df["experiment"].str.startswith("bm25_")].copy()
hybrid_df = results_df[results_df["experiment"].str.startswith("hybrid_")].copy()

best_sparse = sparse_df.iloc[0]["experiment"] if not sparse_df.empty else None
best_hybrid = hybrid_df.iloc[0]["experiment"] if not hybrid_df.empty else None

print("Best sparse:", best_sparse)
print("Best hybrid:", best_hybrid)

Best sparse: bm25_semantic
Best hybrid: hybrid_semantic


## 13) Final QA Phase on Winners Only

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def build_context(results, max_chars=2500):
    context_parts = []
    total = 0
    for r in results:
        txt = r.get("text", "")
        if not txt:
            continue
        if total + len(txt) > max_chars:
            break
        context_parts.append(txt)
        total += len(txt)
    return "\n\n".join(context_parts)

def make_qa_prompt(question, context):
    return (
        "Answer the question using only the context provided below. "
        "Write a complete sentence. If the context is insufficient, say: I don't know.\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )

def is_abstention_answer(answer: str):
    if not isinstance(answer, str):
        return False
    text = answer.strip().lower()
    patterns = [
        "i don't know",
        "i do not know",
        "insufficient context",
        "not enough context",
        "cannot determine",
        "can't determine",
        "unknown",
    ]
    return any(p in text for p in patterns)

# ── QA generator: Groq LLM (full sentences) instead of flan-t5-base (fragments) ──
# flan-t5-base produces keyword-span extractions like "World Cup" or "2022"
# which contain no verifiable factual claims and make faithfulness scoring meaningless.
# Groq generates complete sentence answers that the LLM judge can properly evaluate.
def _init_groq_qa_client():
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret("GROQ_API_KEY")
            if api_key:
                os.environ["GROQ_API_KEY"] = api_key
        except Exception:
            pass
    if not api_key:
        raise EnvironmentError("GROQ_API_KEY required for QA generation.")
    from openai import OpenAI
    return OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

groq_qa_client = _init_groq_qa_client()
QA_GEN_MODEL = "llama-3.1-8b-instant"  # same tier as judge — fast, 500k TPD

def generate_answer_groq(question: str, context: str) -> str:
    """Generate a full-sentence answer via Groq LLM."""
    import time
    from openai import RateLimitError as _RLE
    prompt = make_qa_prompt(question, context)
    for attempt in range(6):
        try:
            resp = groq_qa_client.chat.completions.create(
                model=QA_GEN_MODEL,
                temperature=0,
                max_tokens=150,
                messages=[{"role": "user", "content": prompt}],
            )
            return resp.choices[0].message.content.strip()
        except _RLE:
            if attempt == 5:
                raise
            wait = 2.0 * (2 ** attempt)
            print(f"  QA rate limit – waiting {wait:.0f}s...")
            time.sleep(wait)

# Cross-encoder reranker (still used for retrieval quality)
cross_encoder = init_cross_encoder()

def answer_with_retriever(question, retriever_fn):
    overall_start = time.perf_counter()

    # 1) Retrieve candidate pool + optional reranking latency
    retrieval_start = time.perf_counter()
    candidates = retriever_fn(question, RERANK_CANDIDATES_TOP_K)
    reranked_hits = cross_encoder_rerank(
        query=question,
        candidates=candidates,
        cross_encoder=cross_encoder,
        top_k=QA_CONTEXT_TOP_K,
    )
    retrieval_time_sec = time.perf_counter() - retrieval_start

    # 2) Build context and generate full-sentence answer via Groq
    context = build_context(reranked_hits)
    inference_start = time.perf_counter()
    answer = generate_answer_groq(question, context)
    inference_time_sec = time.perf_counter() - inference_start

    overall_time_sec = time.perf_counter() - overall_start

    return {
        "answer": answer,
        "context": context,
        "reranked_hits": reranked_hits,
        "retrieval_time_sec": retrieval_time_sec,
        "inference_time_sec": inference_time_sec,
        "overall_response_time_sec": overall_time_sec,
    }

winner_map = {}
for name in [best_sparse, best_hybrid]:
    if name is not None and name in experiments:
        winner_map[name] = experiments[name]

qa_eval_questions = evaluation_questions[: min(QA_EVAL_SIZE, len(evaluation_questions))] if DEV_MODE else evaluation_questions

import time
qa_rows = []
qa_artifacts = []
for item in tqdm(qa_eval_questions, desc="QA Comparison (cross-encoder rerank + Groq)"):
    q = item["question"]
    expected_abstain = bool(item.get("expect_abstain", False) or len(item.get("relevant_chunk_ids", [])) == 0)
    row = {"question": q, "expected_abstain": expected_abstain, "question_type": item.get("question_type", "unknown")}
    for winner_name, retriever_fn in winner_map.items():
        result = answer_with_retriever(q, retriever_fn)
        predicted_abstain = is_abstention_answer(result["answer"])
        abstention_correct = (predicted_abstain == expected_abstain)

        row[f"answer_{winner_name}"] = result["answer"]
        row[f"retrieval_time_{winner_name}_sec"] = result["retrieval_time_sec"]
        row[f"inference_time_{winner_name}_sec"] = result["inference_time_sec"]
        row[f"overall_response_time_{winner_name}_sec"] = result["overall_response_time_sec"]
        row[f"predicted_abstain_{winner_name}"] = predicted_abstain
        row[f"abstention_correct_{winner_name}"] = abstention_correct

        qa_artifacts.append({
            "question": q,
            "question_type": item.get("question_type", "unknown"),
            "winner": winner_name,
            "expected_abstain": expected_abstain,
            "predicted_abstain": predicted_abstain,
            "abstention_correct": abstention_correct,
            "answer": result["answer"],
            "context": result["context"],
            "retrieval_time_sec": result["retrieval_time_sec"],
            "inference_time_sec": result["inference_time_sec"],
            "overall_response_time_sec": result["overall_response_time_sec"],
        })
        time.sleep(0.2 if DEV_MODE else 0.4)
    qa_rows.append(row)

qa_df = pd.DataFrame(qa_rows)
qa_artifacts_df = pd.DataFrame(qa_artifacts)

qa_latency_summary_df = (
    qa_artifacts_df
    .groupby("winner", as_index=False)
    .agg(
        avg_retrieval_time_sec=("retrieval_time_sec", "mean"),
        p95_retrieval_time_sec=("retrieval_time_sec", lambda s: s.quantile(0.95)),
        avg_inference_time_sec=("inference_time_sec", "mean"),
        p95_inference_time_sec=("inference_time_sec", lambda s: s.quantile(0.95)),
        avg_overall_response_time_sec=("overall_response_time_sec", "mean"),
        p95_overall_response_time_sec=("overall_response_time_sec", lambda s: s.quantile(0.95)),
    )
    .sort_values("avg_overall_response_time_sec")
    .reset_index(drop=True)
)

qa_quality_latency_summary_df = (
    qa_artifacts_df
    .groupby("winner", as_index=False)
    .agg(
        questions_evaluated=("question", "count"),
        abstention_accuracy=("abstention_correct", "mean"),
        avg_retrieval_time_sec=("retrieval_time_sec", "mean"),
        avg_inference_time_sec=("inference_time_sec", "mean"),
        avg_overall_response_time_sec=("overall_response_time_sec", "mean"),
    )
    .sort_values(["abstention_accuracy", "avg_overall_response_time_sec"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"QA eval questions used: {len(qa_eval_questions)} (DEV_MODE={DEV_MODE})")
qa_quality_latency_summary_df

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

QA Comparison (cross-encoder rerank + Groq): 100%|██████████| 25/25 [03:46<00:00,  9.05s/it]


,question,answer_bm25_semantic,answer_hybrid_semantic
0,What is the FIFA World Cup?,The FIFA World Cup is an international associa...,The FIFA World Cup is an international associa...
1,Which tournament is the 2022 FIFA World Cup?,The 2022 FIFA World Cup was the 22nd FIFA Worl...,The 2022 FIFA World Cup was the 22nd FIFA Worl...
2,Which tournament is the 2018 FIFA World Cup?,The 2018 FIFA World Cup was contested with eig...,The 2018 FIFA World Cup was the 21st FIFA Worl...
3,What is FIFA in world football?,FIFA is a professional football tournament hel...,FIFA is the Fédération Internationale de Footb...
4,What club is AC Milan?,AC Milan is a football club.,AC Milan is an Italian professional football c...


## 14) LLM-as-Judge: Faithfulness and Response Relevancy

Fixed evaluation set: first 15 queries from evaluation_questions.
Faithfulness: claim extraction and verification against retrieved context.
Response relevancy: generate 3 alternate questions from the answer and compute cosine similarity with the original query.

In [ ]:
import time
from openai import OpenAI, RateLimitError as OpenAIRateLimitError

def init_judge_client():
    provider = str(JUDGE_PROVIDER).strip().lower()

    if provider == "groq":
        api_key = os.environ.get("GROQ_API_KEY")
        if not api_key:
            try:
                from kaggle_secrets import UserSecretsClient
                user_secrets = UserSecretsClient()
                secret_value_0 = user_secrets.get_secret("GROQ_API_KEY")
                if secret_value_0:
                    api_key = secret_value_0
                    os.environ["GROQ_API_KEY"] = api_key
            except Exception:
                pass

        if not api_key:
            raise EnvironmentError(
                "GROQ_API_KEY is required for Groq judge evaluation. "
                "Set it in Kaggle Secrets with label GROQ_API_KEY."
            )
        return OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

    if provider == "openai":
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise EnvironmentError(
                "OPENAI_API_KEY is required for OpenAI judge evaluation. "
                "Set it in environment variables or Kaggle Secrets."
            )
        return OpenAI(api_key=api_key)

    raise ValueError("JUDGE_PROVIDER must be either 'groq' or 'openai'.")


def llm_json(client, system_prompt: str, user_prompt: str, model: str = JUDGE_MODEL_NAME,
             max_retries: int = 6, base_delay: float = 2.0):
    """Call the judge LLM with exponential-backoff retry on rate-limit errors."""
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ],
            )
            text = resp.choices[0].message.content
            return json.loads(text)
        except OpenAIRateLimitError:
            if attempt == max_retries - 1:
                raise
            wait = base_delay * (2 ** attempt)
            print(f"  Rate limit hit (attempt {attempt + 1}/{max_retries}). "
                  f"Waiting {wait:.0f}s before retry...")
            time.sleep(wait)
        except Exception:
            raise


def extract_claims(client, answer: str):
    system_prompt = "You extract factual atomic claims from answers."
    user_prompt = (
        "Extract atomic factual claims from the answer. Return JSON with key 'claims' as a list of short strings. "
        "If no factual claim is present, return an empty list.\n\n"
        f"Answer:\n{answer}"
    )
    data = llm_json(client, system_prompt, user_prompt)
    claims = data.get("claims", [])
    if not isinstance(claims, list):
        return []
    return [str(c).strip() for c in claims if str(c).strip()]


def verify_claim(client, question: str, claim: str, context: str):
    system_prompt = "You verify whether a claim is supported by provided context only."
    user_prompt = (
        "Verify if the claim is supported by the context only. "
        "Return JSON with keys: label (supported or unsupported), rationale (1-2 lines), evidence (short quote or empty).\n\n"
        f"Question: {question}\n\n"
        f"Claim: {claim}\n\n"
        f"Context:\n{context}"
    )
    data = llm_json(client, system_prompt, user_prompt)
    label = str(data.get("label", "unsupported")).strip().lower()
    if label not in {"supported", "unsupported"}:
        label = "unsupported"
    return {
        "claim": claim,
        "label": label,
        "rationale": str(data.get("rationale", "")).strip(),
        "evidence": str(data.get("evidence", "")).strip(),
    }


def generate_alternate_questions(client, answer: str):
    system_prompt = "You generate alternative user queries from an answer."
    user_prompt = (
        "Generate exactly 3 different questions that this answer could respond to. "
        "Return JSON with key 'questions' as a list of exactly 3 strings.\n\n"
        f"Answer:\n{answer}"
    )
    data = llm_json(client, system_prompt, user_prompt)
    questions = data.get("questions", [])
    if not isinstance(questions, list):
        questions = []
    questions = [str(q).strip() for q in questions if str(q).strip()]
    return questions[:3]


def cosine_similarity(a: str, b: str, embed_model):
    vecs = embed_model.encode([a, b], normalize_embeddings=True)
    return float(np.dot(vecs[0], vecs[1]))


judge_client = init_judge_client()
relevancy_embed_model = SentenceTransformer(EMBED_MODEL_NAME)

judge_target = best_hybrid if best_hybrid in winner_map else best_sparse
if judge_target is None:
    raise ValueError("No winner retriever found for judge evaluation.")

answerable_judge_questions = [
    item["question"] for item in evaluation_questions
    if not bool(item.get("expect_abstain", False))
]
judge_questions = answerable_judge_questions[: min(JUDGE_EVAL_SIZE, len(answerable_judge_questions))]

judge_input_df = qa_artifacts_df[
    (qa_artifacts_df["winner"] == judge_target) &
    (qa_artifacts_df["question"].isin(judge_questions))
] .copy()

judge_rows = []
example_rows = []

for _, row in tqdm(judge_input_df.iterrows(), total=len(judge_input_df), desc="LLM-as-judge"):
    question = row["question"]
    answer = row["answer"]
    context = row["context"]

    claims = extract_claims(judge_client, answer)
    verifications = []
    for claim in claims:
        verifications.append(verify_claim(judge_client, question, claim, context))
        time.sleep(0.15 if DEV_MODE else 0.3)

    supported = sum(1 for v in verifications if v["label"] == "supported")
    total_claims = len(verifications)
    faithfulness_score = (supported / total_claims) if total_claims > 0 else 0.0

    alt_questions = generate_alternate_questions(judge_client, answer)
    alt_similarities = [cosine_similarity(question, q_alt, relevancy_embed_model) for q_alt in alt_questions]
    avg_relevancy = float(np.mean(alt_similarities)) if alt_similarities else 0.0

    judge_rows.append({
        "question": question,
        "winner": judge_target,
        "num_claims": total_claims,
        "supported_claims": supported,
        "faithfulness_score": faithfulness_score,
        "alternate_questions": json.dumps(alt_questions, ensure_ascii=False),
        "alt_question_similarities": json.dumps(alt_similarities, ensure_ascii=False),
        "average_relevancy_score": avg_relevancy,
    })

    if len(example_rows) < 3:
        example_rows.append({
            "question": question,
            "answer": answer,
            "claims": json.dumps(claims, ensure_ascii=False),
            "verification_results": json.dumps(verifications, ensure_ascii=False),
        })

    time.sleep(0.2 if DEV_MODE else 0.5)

judge_df = pd.DataFrame(judge_rows)
judge_examples_df = pd.DataFrame(example_rows)

print(f"Judge provider: {JUDGE_PROVIDER}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Judge target retriever: {judge_target}")
print(f"Evaluated answerable queries: {len(judge_df)}")
if not judge_df.empty:
    print(f"Average faithfulness score: {judge_df['faithfulness_score'].mean():.4f}")
    print(f"Average response relevancy score: {judge_df['average_relevancy_score'].mean():.4f}")

judge_examples_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
LLM-as-judge: 100%|██████████| 15/15 [04:54<00:00, 19.63s/it]

Judge provider: groq
Judge model: llama-3.1-8b-instant
Judge target retriever: hybrid_semantic
Evaluated queries: 15
Average faithfulness score: 0.9333
Average response relevancy score: 0.8714


,question,answer,claims,verification_results
0,What is the FIFA World Cup?,The FIFA World Cup is an international associa...,"[""The FIFA World Cup is an international assoc...","[{""claim"": ""The FIFA World Cup is an internati..."
1,Which tournament is the 2022 FIFA World Cup?,The 2022 FIFA World Cup was the 22nd FIFA Worl...,"[""The 2022 FIFA World Cup was the 22nd FIFA Wo...","[{""claim"": ""The 2022 FIFA World Cup was the 22..."
2,Which tournament is the 2018 FIFA World Cup?,The 2018 FIFA World Cup was the 21st FIFA Worl...,"[""The 2018 FIFA World Cup was the 21st FIFA Wo...","[{""claim"": ""The 2018 FIFA World Cup was the 21..."


## 15) Ablation Study: Chunking x Retrieval Variations

This ablation compares performance across:
- Chunking: fixed, recursive, semantic
- Retrieval: semantic-only vs hybrid + cross-encoder re-ranking
- Metrics: Faithfulness and Relevancy

In [ ]:
import time

if not dense_ready:
    raise EnvironmentError("Ablation requires dense retrieval. Set PINECONE_API_KEY and rerun retrieval cells.")

JUDGE_MODEL_NAME = "llama-3.1-8b-instant"  # 500k TPD

# ── All 3 chunking strategies × 2 retrieval variants ──────────────────────────
ABLATION_VARIATIONS = [
    ("fixed", "semantic_only"),
    ("fixed", "hybrid_plus_rerank"),
    ("recursive", "semantic_only"),
    ("recursive", "hybrid_plus_rerank"),
    ("semantic", "semantic_only"),
    ("semantic", "hybrid_plus_rerank"),
]

if "judge_client" not in globals():
    judge_client = init_judge_client()
if "relevancy_embed_model" not in globals():
    relevancy_embed_model = SentenceTransformer(EMBED_MODEL_NAME)
if "cross_encoder" not in globals():
    cross_encoder = init_cross_encoder()
if "groq_qa_client" not in globals():
    groq_qa_client = _init_groq_qa_client()

def retrieve_hits_for_ablation(question: str, chunking: str, retrieval_variant: str):
    if retrieval_variant == "semantic_only":
        return dense_retrieve(
            query=question,
            namespace=chunking,
            strategy=chunking,
            index=index,
            embed_model=embed_model,
            top_k=QA_CONTEXT_TOP_K,
        )
    if retrieval_variant == "hybrid_plus_rerank":
        hybrid_candidates = hybrid_retrieve(
            query=question,
            strategy=chunking,
            namespace=chunking,
            index=index,
            embed_model=embed_model,
            top_k=RERANK_CANDIDATES_TOP_K,
        )
        return cross_encoder_rerank(
            query=question,
            candidates=hybrid_candidates,
            cross_encoder=cross_encoder,
            top_k=QA_CONTEXT_TOP_K,
        )
    raise ValueError(f"Unknown retrieval_variant: {retrieval_variant}")

ablation_source_questions = [
    item["question"] for item in evaluation_questions
    if not bool(item.get("expect_abstain", False))
]
ablation_questions = ablation_source_questions[: min(ABLATION_EVAL_SIZE, len(ablation_source_questions))]
ablation_rows = []

for question in tqdm(ablation_questions, desc="Ablation (3 chunking × 2 retrieval variants)"):
    for chunking, retrieval_variant in ABLATION_VARIATIONS:
        overall_start = time.perf_counter()

        retrieval_start = time.perf_counter()
        selected_hits = retrieve_hits_for_ablation(question, chunking, retrieval_variant)
        retrieval_time_sec = time.perf_counter() - retrieval_start

        context = build_context(selected_hits)

        inference_start = time.perf_counter()
        answer = generate_answer_groq(question, context)  # full-sentence via Groq
        inference_time_sec = time.perf_counter() - inference_start

        overall_response_time_sec = time.perf_counter() - overall_start
        time.sleep(0.2 if DEV_MODE else 0.3)

        claims = extract_claims(judge_client, answer)
        time.sleep(0.2 if DEV_MODE else 0.3)

        verifications = []
        for claim in claims:
            verifications.append(verify_claim(judge_client, question, claim, context))
            time.sleep(0.2 if DEV_MODE else 0.3)

        supported = sum(1 for v in verifications if v["label"] == "supported")
        total_claims = len(verifications)
        faithfulness_score = (supported / total_claims) if total_claims > 0 else 0.0

        alt_questions = generate_alternate_questions(judge_client, answer)
        alt_similarities = [
            cosine_similarity(question, q_alt, relevancy_embed_model) for q_alt in alt_questions
        ]
        average_relevancy_score = float(np.mean(alt_similarities)) if alt_similarities else 0.0

        ablation_rows.append({
            "question": question,
            "chunking": chunking,
            "retrieval_variant": retrieval_variant,
            "answer": answer,
            "num_claims": total_claims,
            "supported_claims": supported,
            "faithfulness_score": faithfulness_score,
            "average_relevancy_score": average_relevancy_score,
            "retrieval_time_sec": retrieval_time_sec,
            "inference_time_sec": inference_time_sec,
            "overall_response_time_sec": overall_response_time_sec,
        })
        time.sleep(0.2 if DEV_MODE else 0.5)

ablation_detail_df = pd.DataFrame(ablation_rows)
ablation_comparison_df = (
    ablation_detail_df
    .groupby(["chunking", "retrieval_variant"], as_index=False)
    .agg(
        queries=("question", "count"),
        faithfulness=("faithfulness_score", "mean"),
        relevancy=("average_relevancy_score", "mean"),
        avg_retrieval_time_sec=("retrieval_time_sec", "mean"),
        p95_retrieval_time_sec=("retrieval_time_sec", lambda s: s.quantile(0.95)),
        avg_inference_time_sec=("inference_time_sec", "mean"),
        p95_inference_time_sec=("inference_time_sec", lambda s: s.quantile(0.95)),
        avg_overall_response_time_sec=("overall_response_time_sec", "mean"),
        p95_overall_response_time_sec=("overall_response_time_sec", lambda s: s.quantile(0.95)),
    )
    .sort_values(["faithfulness", "relevancy", "avg_overall_response_time_sec"], ascending=[False, False, True])
    .reset_index(drop=True)
)

ablation_comparison_df["throughput_qps"] = 1.0 / ablation_comparison_df["avg_overall_response_time_sec"].clip(lower=1e-9)

print(f"Ablation model: {JUDGE_MODEL_NAME}")
print(f"Questions evaluated: {len(ablation_questions)} (DEV_MODE={DEV_MODE})")
print(f"Variation count: {len(ABLATION_VARIATIONS)}")
ablation_comparison_df

Ablation (3 chunking × 2 retrieval variants): 100%|██████████| 15/15 [33:56<00:00, 135.75s/it]

Ablation model: llama-3.1-8b-instant
Questions evaluated: 15
Variation count: 6


,chunking,retrieval_variant,queries,faithfulness,relevancy
0,fixed,semantic_only,15,0.977778,0.883466
1,recursive,semantic_only,15,0.977778,0.883466
2,recursive,hybrid_plus_rerank,15,0.977778,0.859881
3,fixed,hybrid_plus_rerank,15,0.977778,0.859292
4,semantic,semantic_only,15,0.955556,0.875956
5,semantic,hybrid_plus_rerank,15,0.933333,0.868358


## 16) Save Outputs

In [ ]:
def resolve_output_dir(project_root: Path) -> Path:
    candidates = [
        Path("/kaggle/working") / "outputs",
        Path.cwd() / "outputs",
        project_root / "outputs",
    ]

    for candidate in candidates:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate
        except OSError:
            continue

    raise OSError("Could not create a writable outputs directory.")

output_dir = resolve_output_dir(PROJECT_ROOT)

if 'evaluation_questions_df' in globals():
    evaluation_questions_df.to_csv(output_dir / "evaluation_question_bank.csv", index=False)

if 'results_df' in globals():
    results_df.to_csv(output_dir / "retrieval_metrics.csv", index=False)

if 'qa_df' in globals():
    qa_df.to_csv(output_dir / "qa_winner_comparison.csv", index=False)

if 'qa_artifacts_df' in globals():
    qa_artifacts_df.to_csv(output_dir / "qa_artifacts.csv", index=False)

if 'qa_latency_summary_df' in globals():
    qa_latency_summary_df.to_csv(output_dir / "qa_latency_summary.csv", index=False)

if 'qa_quality_latency_summary_df' in globals():
    qa_quality_latency_summary_df.to_csv(output_dir / "qa_quality_latency_summary.csv", index=False)

if 'judge_df' in globals():
    judge_df.to_csv(output_dir / "llm_judge_metrics.csv", index=False)

if 'judge_examples_df' in globals():
    judge_examples_df.to_csv(output_dir / "llm_judge_examples.csv", index=False)

if 'ablation_detail_df' in globals():
    ablation_detail_df.to_csv(output_dir / "ablation_detail_metrics.csv", index=False)

if 'ablation_comparison_df' in globals():
    ablation_comparison_df.to_csv(output_dir / "ablation_comparison_metrics.csv", index=False)

print(f"Saved outputs in: {output_dir}")

Saved outputs in: /kaggle/working/outputs
